In [2]:
import pandas as pd
import os
import sys


class DataCleaner:
    def __init__(self, file_path):
        """
        初始化：读取文件
        :param file_path: CSV 或 Excel 文件路径
        """
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件不存在: {file_path}")

        self.file_path = file_path
        self.file_ext = os.path.splitext(file_path)[1].lower()
        self.df = None

        print(f"正在读取文件: {file_path} ...")
        try:
            if self.file_ext == '.csv':
                self.df = pd.read_csv(file_path)
            elif self.file_ext in ['.xlsx', '.xls']:
                self.df = pd.read_excel(file_path)
            else:
                raise ValueError("不支持的文件格式，请使用 .csv 或 .excel (.xlsx/.xls)")

            print(f"读取成功！原始数据形状: {self.df.shape}")
            print(self.df.info())
        except Exception as e:
            print(f"读取文件失败: {e}")
            sys.exit(1)

    def handle_missing_values(self, strategy='drop', fill_value=None):
        """
        处理缺失值
        :param strategy: 'drop' (删除), 'fill_mean' (均值填充数值列), 'fill_value' (指定值填充)
        :param fill_value: 当 strategy 为 'fill_value' 时使用的具体值
        """
        initial_count = self.df.isnull().sum().sum()
        if initial_count == 0:
            print("未发现缺失值。")
            return

        print(f"发现 {initial_count} 个缺失值，正在执行策略: {strategy}...")

        if strategy == 'drop':
            # 删除包含任何缺失值的行
            self.df.dropna(inplace=True)
        elif strategy == 'fill_mean':
            # 仅对数值列进行均值填充
            numeric_cols = self.df.select_dtypes(include=['number']).columns
            self.df[numeric_cols] = self.df[numeric_cols].fillna(self.df[numeric_cols].mean())
            # 非数值列如果还有缺失，可以选择删除或填默认值，这里简单起见填 'Unknown'
            non_numeric_cols = self.df.select_dtypes(exclude=['number']).columns
            self.df[non_numeric_cols] = self.df[non_numeric_cols].fillna('Unknown')
        elif strategy == 'fill_value':
            if fill_value is None:
                fill_value = 0  # 默认填0
            self.df.fillna(fill_value, inplace=True)

        remaining_count = self.df.isnull().sum().sum()
        print(f"缺失值处理完成。剩余缺失值: {remaining_count}")

    def remove_duplicates(self):
        """
        删除完全重复的行
        """
        initial_rows = len(self.df)
        self.df.drop_duplicates(inplace=True)
        removed_count = initial_rows - len(self.df)
        print(f"删除重复值完成。共删除 {removed_count} 行重复数据。")

    def convert_formats(self, column_rules):
        """
        数据格式转换
        :param column_rules: 字典，格式 {列名: 目标类型}
        例如: {'age': 'int', 'price': 'float', 'date': 'datetime', 'name': 'str'}
        """
        print("正在执行格式转换...")
        for col, dtype in column_rules.items():
            if col not in self.df.columns:
                print(f"警告: 列 '{col}' 不存在，跳过。")
                continue

            try:
                if dtype == 'datetime':
                    self.df[col] = pd.to_datetime(self.df[col], errors='coerce')
                elif dtype == 'int':
                    # 先转float再转int，防止有NaN报错，或者先处理缺失值
                    self.df[col] = pd.to_numeric(self.df[col], errors='coerce').astype('Int64')  # Int64支持NaN
                elif dtype == 'float':
                    self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
                elif dtype == 'str':
                    self.df[col] = self.df[col].astype(str)

                print(f"列 '{col}' 已转换为 {dtype}")
            except Exception as e:
                print(f"列 '{col}' 转换失败: {e}")

    def save_data(self, output_path):
        """
        保存清洗后的数据
        :param output_path: 输出文件路径
        """
        ext = os.path.splitext(output_path)[1].lower()
        try:
            if ext == '.csv':
                self.df.to_csv(output_path, index=False)
            elif ext in ['.xlsx', '.xls']:
                self.df.to_excel(output_path, index=False)
            else:
                raise ValueError("输出格式必须是 .csv 或 .xlsx")

            print(f"清洗完成！数据已保存至: {output_path}")
            print(f"最终数据形状: {self.df.shape}")
        except Exception as e:
            print(f"保存文件失败: {e}")